Summary table for business question 4.1. Pre-aggregating sales by region and
trade group turns the ranking query into a scan of a handful of rows instead
of a full pass over the fact.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'gold'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'agg_sales_region_trade_group'

In [0]:
df_fact_sales = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.fact_sales').alias('fs')
df_dim_region = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_region').alias('dr')
df_dim_channel = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_channel').alias('dc')

In [0]:
df_agg = (
    df_fact_sales
    .join(
        F.broadcast(df_dim_region),
        on='region_key',
        how='inner'
    )
    .join(
        F.broadcast(df_dim_channel),
        on='channel_key',
        how='inner'
    )
    .groupBy(
        'dr.region',
        'dc.trade_group'
    )
    .agg(
        F.sum('fs.dollar_volume').alias('dollar_volume'),
        F.count('*').alias('record_count'),
        F.countDistinct('dc.trade_channel').alias('channel_count')
    )
)

In [0]:
df_agg\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')